# ARS Notebook 3 — Mathematical OOD Evaluation

Four-arm paired evaluation on GSM-Plus, SVAMP, and ASDiv. GSM-Plus probes controlled variations related to GSM8K; SVAMP and ASDiv test transfer across independently collected math-word-problem sets and broader language/problem patterns. Each dataset is capped at 500 test questions and evaluated in 0-shot and 2-shot. The actual paired count and source IDs are saved in each record.

## Input contract

Attach one complete `ars_training_bundle.zip` or its extracted Kaggle Model input with the root `manifest.json`. The bundle must contain the fixed adapter, route bank, router, both norm calibrations, and static action. The numeric answer scorer is used only for numeric datasets. The public EleutherAI/ASDiv release has one 2,305-row `validation` split and no official test split; this notebook reports that split transparently as an external evaluation pool. MATH-500 remains excluded until a validated symbolic-equivalence grader is available.

In [ ]:
import hashlib, json, os, shutil, subprocess, sys
from pathlib import Path

os.environ["TOKENIZERS_PARALLELISM"] = "false"
REPO_URL = "https://github.com/Abdullah182155/adaptive-residual-steering.git"
REPO_DIR = Path("/kaggle/working/adaptive-residual-steering")
WORK_ROOT = Path("/kaggle/working")
INPUT_ROOT = Path("/kaggle/input")
if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", "development"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "development"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", "development"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", "development", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
from scripts.research_notebook_utils import (
    extract_zip_once, locate_named_file, merge_result_bundle, pack_files,
    resolve_artifacts, run_live, sha256_file, atomic_write_json,
)

# Keep the training and evaluation revisions explicit. A stale script snapshot
# inside the training bundle must not replace the current benchmark runner.
EVALUATION_CODE_COMMIT = subprocess.run(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
    check=True, capture_output=True, text=True,
).stdout.strip()
training_zips = sorted(INPUT_ROOT.rglob("ars_training_bundle.zip"))
if len(training_zips) > 1:
    raise ValueError(f"Attach one training bundle only; found: {[str(p) for p in training_zips]}")
TRAINING_ZIP = training_zips[0] if training_zips else None
TRAINING_INPUT = None
TRAINING_MANIFEST = {}
if TRAINING_ZIP:
    ARTIFACT_ROOT = extract_zip_once(TRAINING_ZIP, WORK_ROOT / "ars_training_artifacts")
    TRAINING_INPUT = TRAINING_ZIP
    manifest_path = ARTIFACT_ROOT / "manifest.json"
    if manifest_path.is_file():
        TRAINING_MANIFEST = json.loads(manifest_path.read_text(encoding="utf-8"))
else:
    manifests = []
    for candidate in INPUT_ROOT.rglob("manifest.json"):
        try:
            data = json.loads(candidate.read_text(encoding="utf-8"))
        except (OSError, ValueError, TypeError):
            continue
        if data.get("format") == "ars_training_bundle_v1":
            manifests.append((candidate, data))
    if len(manifests) > 1:
        raise ValueError(f"Attach one extracted ARS training bundle only; found: {[str(p) for p, _ in manifests]}")
    if manifests:
        manifest_path, TRAINING_MANIFEST = manifests[0]
        ARTIFACT_ROOT = manifest_path.parent
        TRAINING_INPUT = ARTIFACT_ROOT
    else:
        ARTIFACT_ROOT = INPUT_ROOT
        manifest_path = None

REQUIRED_ARTIFACTS = [
    "fixed_route_adapter.pt", "session3_steering_bank.pt", "session3_router.pt",
    "fixed_route_p99_norms.json", "candidate_norms_p99.json",
    "session3_static_bank_action.json",
]
try:
    ARTIFACTS = resolve_artifacts(ARTIFACT_ROOT, REQUIRED_ARTIFACTS)
    INPUTS_READY = bool(TRAINING_INPUT)
except Exception as exc:
    ARTIFACTS, INPUTS_READY = {}, False
    print("Training bundle is missing or invalid:", exc, flush=True)
if not TRAINING_INPUT:
    INPUTS_READY = False
    print("Attach ars_training_bundle.zip or one extracted input containing its root manifest.json.", flush=True)
elif (
    not TRAINING_MANIFEST.get("training_ready", False)
    or TRAINING_MANIFEST.get("architecture_id") != "ars.fixed_route_low_rank_residual"
    or TRAINING_MANIFEST.get("architecture_version") != "1.0.0"
):
    INPUTS_READY = False
    print("Training bundle is not ready or has an unsupported fixed-adapter architecture.", TRAINING_MANIFEST.get("required_phase_status"), flush=True)
TRAINING_CODE_COMMIT = TRAINING_MANIFEST.get("code_commit")
TRAINING_INPUT_SHA256 = (
    sha256_file(TRAINING_ZIP) if TRAINING_ZIP else
    sha256_file(manifest_path) if TRAINING_INPUT and manifest_path and manifest_path.is_file() else None
)
EXPECTED_ARTIFACT_SHA256 = (
    {
        "fixed_adapter_checkpoint": sha256_file(ARTIFACTS["fixed_route_adapter.pt"]),
        "fixed_adapter_norms": sha256_file(ARTIFACTS["fixed_route_p99_norms.json"]),
        "bank_checkpoint": sha256_file(ARTIFACTS["session3_steering_bank.pt"]),
        "router_checkpoint": sha256_file(ARTIFACTS["session3_router.pt"]),
        "candidate_norms": sha256_file(ARTIFACTS["candidate_norms_p99.json"]),
        "static_bank_action": sha256_file(ARTIFACTS["session3_static_bank_action.json"]),
    }
    if INPUTS_READY else {}
)
ARTIFACT_FINGERPRINT = hashlib.sha256(
    json.dumps(EXPECTED_ARTIFACT_SHA256, sort_keys=True).encode("utf-8")
).hexdigest()[:12] if INPUTS_READY else "missing"
print({
    "repository": str(REPO_DIR),
    "evaluation_code_commit": EVALUATION_CODE_COMMIT,
    "training_code_commit": TRAINING_CODE_COMMIT,
    "training_input": str(TRAINING_INPUT) if TRAINING_INPUT else None,
    "input_mode": "zip" if TRAINING_ZIP else ("extracted_directory" if TRAINING_INPUT else "missing"),
    "artifacts_ready": INPUTS_READY,
    "artifact_fingerprint": ARTIFACT_FINGERPRINT,
    "artifacts": {key: str(value) for key, value in ARTIFACTS.items()},
    "gpus": subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.splitlines(),
}, flush=True)


## Choose one dataset for this session

Options: `gsm_plus`, `svamp`, `asdiv`. Run one dataset per Kaggle session. Both 0-shot and 2-shot use the same seeded test IDs, and all four arms use identical prompts within each condition. ASDiv has 2,305 items in the selected public release; this notebook samples up to 500.

In [ ]:
SEED = 42
N_PER_DATASET = 500
SHOTS = [0, 2]
DATASET_THIS_SESSION = "gsm_plus"
from src.evaluation.ood import SPECS as OOD_SPECS, load_ood_rows
_current_ood_rows = load_ood_rows(DATASET_THIS_SESSION)
CURRENT_DATASET_METADATA = {"dataset_id": OOD_SPECS[DATASET_THIS_SESSION][0],
    "split": _current_ood_rows[0].get("source_split"),
    "fingerprint": _current_ood_rows[0].get("dataset_fingerprint"),
    "available_rows": len(_current_ood_rows), "sampling_seed": SEED}
SUPPORTED_DATASETS = {"gsm_plus", "svamp", "asdiv"}
if DATASET_THIS_SESSION not in SUPPORTED_DATASETS:
    raise ValueError(f"Choose one of {sorted(SUPPORTED_DATASETS)}")
if not INPUTS_READY:
    print("Evaluation is blocked until a complete training bundle is attached.", flush=True)
RESULTS = WORK_ROOT / "ars_math_ood_results"
RESULTS.mkdir(parents=True, exist_ok=True)
previous_bundles = sorted(INPUT_ROOT.rglob("ars_math_ood_results.zip"))
if len(previous_bundles) > 1:
    raise ValueError(f"Attach one previous result bundle only; found: {[str(p) for p in previous_bundles]}")
if previous_bundles:
    merge_result_bundle(previous_bundles[0], RESULTS)

def is_done(dataset, shot):
    if not INPUTS_READY:
        return False
    for path in RESULTS.rglob(f"session3_{dataset}_{N_PER_DATASET}_seed{SEED}.json"):
        try:
            record = json.loads(path.read_text(encoding="utf-8"))
            if (
                record.get("suite") == dataset
                and record.get("test_seed") == SEED
                and len(record.get("test_source_indices", [])) == N_PER_DATASET
                and set(record.get("shots", {})) == {str(shot)}
                and record.get("four_arm_complete")
                and record.get("artifact_sha256") == EXPECTED_ARTIFACT_SHA256
                and record.get("provenance", {}).get("evaluation_code_commit") == EVALUATION_CODE_COMMIT
                and record.get("dataset_metadata") == CURRENT_DATASET_METADATA
            ):
                return True
        except (OSError, ValueError, TypeError):
            continue
    return False

def run_condition(dataset, shot):
    if is_done(dataset, shot):
        print("COMPLETE for these exact artifacts; skip", dataset, shot, flush=True)
        return
    if not INPUTS_READY:
        print("SKIP: missing or invalid training bundle", flush=True)
        return
    out = RESULTS / ARTIFACT_FINGERPRINT / EVALUATION_CODE_COMMIT[:12] / dataset / f"{shot}shot_n{N_PER_DATASET}"
    out.mkdir(parents=True, exist_ok=True)
    shutil.copy2(ARTIFACTS["session3_static_bank_action.json"], out / "session3_static_bank_action.json")
    cmd = [sys.executable, "-u", "scripts/run_session3_router.py", "--stage", "benchmark",
           "--skip_gsm8k", "--ood_datasets", dataset, "--ood_n", str(N_PER_DATASET),
           "--output_dir", str(out), "--candidate_layers", "8", "11", "12", "13", "15", "17", "18", "19", "20", "21", "22", "23", "24", "26",
           "--norm_calibration", str(ARTIFACTS["candidate_norms_p99.json"]),
           "--fixed_adapter_norm_calibration", str(ARTIFACTS["fixed_route_p99_norms.json"]),
           "--fixed_adapter_checkpoint", str(ARTIFACTS["fixed_route_adapter.pt"]),
           "--bank_checkpoint", str(ARTIFACTS["session3_steering_bank.pt"]),
           "--router_checkpoint", str(ARTIFACTS["session3_router.pt"]),
           "--bench_seed", str(SEED), "--shots", str(shot), "--require_all_arms", "--diversify_prompts"]
    code = run_live(cmd, cwd=REPO_DIR, log_path=out / "benchmark.log")
    record_path = out / f"session3_{dataset}_{N_PER_DATASET}_seed{SEED}.json"
    if code == 0 and record_path.is_file():
        record = json.loads(record_path.read_text(encoding="utf-8"))
        record["provenance"] = {
            "training_code_commit": TRAINING_CODE_COMMIT,
            "evaluation_code_commit": EVALUATION_CODE_COMMIT,
            "training_input_sha256": TRAINING_INPUT_SHA256,
            "artifact_fingerprint": ARTIFACT_FINGERPRINT,
            "primary_shots": [0, 2],
        }
        atomic_write_json(record_path, record)
    print({"dataset": dataset, "shot": shot, "exit_code": code, "complete": is_done(dataset, shot)}, flush=True)

for shot in SHOTS:
    run_condition(DATASET_THIS_SESSION, shot)

from IPython.display import display
import pandas as pd
from scripts.research_notebook_analysis import build_benchmark_summary

metrics_summary = build_benchmark_summary(
    [RESULTS], RESULTS / "analysis",
    artifact_sha256=EXPECTED_ARTIFACT_SHA256,
    artifact_fingerprint=ARTIFACT_FINGERPRINT,
    evaluation_code_commit=EVALUATION_CODE_COMMIT,
)
metrics = metrics_summary["arm_metrics"]
if metrics:
    metrics_table = pd.DataFrame(metrics)
    performance_columns = [
        "suite", "shot", "arm", "n", "accuracy_pct",
        "paired_delta_pp_vs_baseline", "paired_ci95_pp_vs_baseline", "mcnemar_p_value_vs_baseline",
        "n_timed", "timing_coverage_rate_pct", "mean_generation_latency_seconds", "median_generation_latency_seconds",
        "p90_generation_latency_seconds", "total_generation_seconds",
        "questions_per_second", "output_tokens_per_second", "mean_prompt_tokens", "mean_new_tokens", "timed_output_tokens",
        "router_unique_routes", "router_most_used_route_share", "router_k_counts", "device_name",
    ]
    quality_columns = [
        "suite", "shot", "arm", "n", "strict_accuracy_pct", "marker_accuracy_pct",
        "lenient_accuracy_pct", "extraction_rate_pct", "max_new_tokens_rate_pct", "stop_reason_counts",
        "equation_coverage_rate_pct", "mean_eaa_when_equations_present",
        "clean_arithmetic_rate_pct", "guard_trip_rate_pct", "nonfinite_count",
    ]
    pd.set_option("display.max_columns", None)
    pd.set_option("display.width", 2200)
    print("Paired accuracy and performance (latency in seconds; rates are per model-generation call):")
    display(metrics_table[[name for name in performance_columns if name in metrics_table]].round(3))
    print("Answer quality, output format, arithmetic diagnostics, and safety checks:")
    display(metrics_table[[name for name in quality_columns if name in metrics_table]].round(3))
    paired_table = pd.DataFrame(metrics_summary["paired_comparisons"])
    if not paired_table.empty:
        print("Paired arm-to-arm comparisons:")
        display(paired_table[[name for name in ["suite", "shot", "reference_arm", "arm", "n", "wins", "losses", "ties", "delta_pp", "ci95_pp", "mcnemar_p_value"] if name in paired_table]].round(4))
    route_table = pd.DataFrame(metrics_summary["route_usage"])
    if not route_table.empty:
        print("Router route, k, layer, and gain usage:")
        display(route_table)
else:
    print("No complete four-arm records for the current artifacts and evaluation-code revision yet.")
print({
    "summary_json": str(RESULTS / "analysis" / "benchmark_metrics_summary.json"),
    "summary_csv": str(RESULTS / "analysis" / "benchmark_metrics_summary.csv"),
    "paired_comparisons_csv": str(RESULTS / "analysis" / "benchmark_paired_comparisons.csv"),
    "route_usage_csv": str(RESULTS / "analysis" / "benchmark_route_usage.csv"),
    "matched_records": metrics_summary["record_count"],
    "measurement_definitions": metrics_summary["measure_definitions"],
}, flush=True)

files = {p: str(p.relative_to(RESULTS)) for p in RESULTS.rglob("*") if p.is_file() and p.name != ".source_sha256"}
for name in ("run_session3_router.py", "run_session3_mcq_benchmark.py", "research_notebook_utils.py", "research_notebook_analysis.py"):
    path = REPO_DIR / "scripts" / name
    archive_name = f"code/scripts/{name}"
    if path.is_file() and archive_name not in files.values():
        files[path] = archive_name
for relative in ("src/evaluation/benchmark.py", "src/evaluation/answers.py", "src/evaluation/ood.py"):
    path = REPO_DIR / relative
    archive_name = f"code/{relative}"
    if path.is_file() and archive_name not in files.values():
        files[path] = archive_name
zip_path = pack_files(files, WORK_ROOT / "ars_math_ood_results.zip") if files else None
print({"dataset": DATASET_THIS_SESSION, "result_bundle": str(zip_path) if zip_path else None,
       "artifact_fingerprint": ARTIFACT_FINGERPRINT}, flush=True)


## Interpretation

Report each dataset separately. GSM-Plus is not a fully independent domain. Do not pool its score with MATH-500 or SVAMP into a single OOD accuracy.
The final cell displays accuracy/performance, quality, all paired arm comparisons, and router-use tables. The archive includes `analysis/benchmark_metrics_summary.csv`, `benchmark_paired_comparisons.csv`, `benchmark_route_usage.csv`, and `benchmark_metrics_summary.json`; every arm record retains per-question timing and token counts.